# 11.32 — Sim-to-Real Transfer

Sim-to-real transfer asks a practical RL question: how can a policy trained in a cheap simulator keep working when the real world has different friction, mass, delay, sensor noise, or actuation strength? In this lesson, we build the idea from scratch with tiny NumPy control problems, treating the simulator parameter as part of the learning problem rather than pretending it is exactly correct.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build sim-to-real transfer one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math, including the reality gap, domain randomization, and robustness objective, is made visible. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, random sampling, and small control calculations.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for every sampled simulator domain.

### 1. The reality gap: one policy, two dynamics

A simulator is a model of the world, not the world itself. We use a one-dimensional toy robot whose state is its position error `x`; the policy applies an action `a = -k x` to push the error back toward zero. The simulator says the next error is `x' = x + actuator * a`, but the real actuator may be weaker. That difference is the **reality gap**: the learned action is correct for the simulated parameter and slightly wrong for the real one.

In [ ]:
x0_w = 1.0  # start one unit away from the target.
k_w = 1.0  # feedback gain: action is -k*x.
sim_actuator_w = 1.0  # simulator believes actions have full strength.
real_actuator_w = 0.72  # real robot is weaker than the simulator.
print("start error:", x0_w, "gain:", k_w)
print("sim actuator:", sim_actuator_w, "real actuator:", real_actuator_w)

▶ What you'll see: the same feedback gain is paired with two different transition parameters.

In [ ]:
a_w = -k_w * x0_w  # action chosen by the policy.
x_next_sim_w = x0_w + sim_actuator_w * a_w  # simulator transition.
x_next_real_w = x0_w + real_actuator_w * a_w  # real transition with mismatch.
print("action:", round(a_w, 3))
print("next error in sim:", round(x_next_sim_w, 3), "next error in real:", round(x_next_real_w, 3))
assert round(x_next_sim_w, 3) == 0.000 and round(x_next_real_w, 3) == 0.280

▶ What you'll see: the simulator reaches zero error in one step, while the real system still has error 0.28.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["initial", "sim next", "real next"], [x0_w, x_next_sim_w, x_next_real_w], color=["gray", "seagreen", "crimson"])
plt.ylabel("position error")
plt.title("1: the reality gap leaves residual error")
plt.show()

▶ What you'll see: a perfect simulated correction and an imperfect real correction side by side.

*Why it's done this way:* the transition equation exposes exactly where transfer can fail. The policy maps state to action, but the world maps action to consequence; if `actuator` changes, the same action produces a different next state. Sim-to-real therefore optimizes for consequences under uncertainty, not just for high simulator reward.

### 2. Training in one nominal simulator overfits the assumed parameter

For this toy system, the one-step squared-error loss is `(x + actuator*(-k*x))² = x²(1 - actuator*k)²`. In a simulator with actuator strength 1, the best gain is `k = 1`. That is mathematically perfect in the simulator, but it is only perfect because the simulator parameter was treated as certain.

In [ ]:
grid_w = np.linspace(0.0, 1.8, 181)  # candidate feedback gains.
x_train_w = 1.0  # evaluate one-step correction from unit error.
sim_loss_w = (x_train_w + sim_actuator_w * (-grid_w * x_train_w)) ** 2  # squared next error in sim.
best_k_sim_w = float(grid_w[np.argmin(sim_loss_w)])
print("best simulator gain:", round(best_k_sim_w, 3))
assert round(best_k_sim_w, 3) == 1.000

▶ What you'll see: the nominal simulator chooses gain 1.0 because that cancels error when actuator = 1.

In [ ]:
real_loss_for_sim_k_w = (x_train_w + real_actuator_w * (-best_k_sim_w * x_train_w)) ** 2
print("real squared error using simulator-optimal k:", round(real_loss_for_sim_k_w, 4))
assert round(real_loss_for_sim_k_w, 4) == 0.0784

▶ What you'll see: a policy with zero simulator loss has nonzero real loss under weaker actuation.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(grid_w, sim_loss_w, color="teal", label="sim loss")
plt.axvline(best_k_sim_w, color="black", linestyle="--", label="sim optimum")
plt.scatter([best_k_sim_w], [real_loss_for_sim_k_w], color="crimson", label="real loss at sim optimum")
plt.xlabel("gain k")
plt.ylabel("one-step squared error")
plt.title("2: nominal training optimizes one world")
plt.legend()
plt.show()

▶ What you'll see: the simulator loss bottoms out at k=1, while the red real-loss point is not zero.

*Why it's done this way:* optimizing a single simulator solves `min_k L(k; θ_sim)`. Transfer asks about `L(k; θ_real)`. Those are the same only if `θ_sim = θ_real`; otherwise the optimum can be biased toward the simulator's mistaken physics.

### 3. Domain randomization trains on a distribution of worlds

Domain randomization replaces one simulator parameter with a range: sample actuator strengths, frictions, delays, or sensor noise each episode. The objective becomes the expected loss `E_θ[L(k; θ)]`, so a gain must work across many plausible worlds. This usually sacrifices perfection in the nominal simulator to reduce failure under mismatch.

In [ ]:
actuators_w = np.linspace(0.6, 1.2, 301)  # plausible simulator domains.
losses_by_k_w = []
for k_try_w in grid_w:
    next_errors_w = x_train_w + actuators_w * (-k_try_w * x_train_w)
    losses_by_k_w.append(np.mean(next_errors_w ** 2))
rand_loss_w = np.array(losses_by_k_w)
best_k_rand_w = float(grid_w[np.argmin(rand_loss_w)])
print("domain-randomized best gain:", round(best_k_rand_w, 3))
assert 1.05 <= best_k_rand_w <= 1.09

▶ What you'll see: the best randomized gain is above 1.0 because the randomized interval includes many weak actuators that need stronger correction.

In [ ]:
sim_loss_rand_k_w = (1 + sim_actuator_w * (-best_k_rand_w)) ** 2
real_loss_rand_k_w = (1 + real_actuator_w * (-best_k_rand_w)) ** 2
print("sim loss at randomized k:", round(sim_loss_rand_k_w, 4))
print("real loss at randomized k:", round(real_loss_rand_k_w, 4))
assert real_loss_rand_k_w < real_loss_for_sim_k_w

▶ What you'll see: the randomized gain is slightly worse in the nominal simulator but better on the weak real actuator.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.plot(grid_w, rand_loss_w, color="purple", label="average randomized loss")
plt.axvline(best_k_sim_w, color="gray", linestyle="--", label="nominal k")
plt.axvline(best_k_rand_w, color="purple", linestyle="--", label="randomized k")
plt.xlabel("gain k")
plt.ylabel("mean squared next error")
plt.title("3: optimize average performance over domains")
plt.legend()
plt.show()

▶ What you'll see: averaging over many actuator strengths shifts the optimum toward the gain that best balances weak under-correction against strong over-correction.

*Why it's done this way:* the expectation over randomized domains is a robustness prior. A gain that is perfect when actuation is exactly 1 can under-correct weaker worlds and over-correct stronger worlds; averaging penalizes both errors, so the chosen gain balances the whole parameter interval.

### 4. Robustness means checking an entire transfer curve

A transfer claim should not be judged by one lucky real parameter. We evaluate each policy across a sweep of possible real actuator strengths and inspect the curve. The robust policy is the one with smaller loss over the plausible deployment region, even if it does not win at every single point.

In [ ]:
real_grid_w = np.linspace(0.5, 1.3, 161)  # possible deployment actuators.
loss_nominal_curve_w = (1 + real_grid_w * (-best_k_sim_w)) ** 2
loss_random_curve_w = (1 + real_grid_w * (-best_k_rand_w)) ** 2
mean_nominal_w = float(np.mean(loss_nominal_curve_w))
mean_random_w = float(np.mean(loss_random_curve_w))
print("mean loss nominal:", round(mean_nominal_w, 4), "randomized:", round(mean_random_w, 4))
assert mean_random_w < mean_nominal_w

▶ What you'll see: the randomized policy has lower average deployment loss across the sweep.

In [ ]:
worst_nominal_w = float(np.max(loss_nominal_curve_w))
worst_random_w = float(np.max(loss_random_curve_w))
print("worst loss nominal:", round(worst_nominal_w, 4), "randomized:", round(worst_random_w, 4))
assert worst_random_w < worst_nominal_w

▶ What you'll see: the robust policy also improves the worst case over this actuator interval.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(real_grid_w, loss_nominal_curve_w, label="trained at actuator=1", color="crimson")
plt.plot(real_grid_w, loss_random_curve_w, label="domain randomized", color="seagreen")
plt.axvline(real_actuator_w, color="black", linestyle="--", label="example real")
plt.xlabel("real actuator strength")
plt.ylabel("one-step squared error")
plt.title("4: transfer is a curve, not one score")
plt.legend()
plt.show()

▶ What you'll see: the randomized curve is flatter across the plausible real region.

*Why it's done this way:* robustness is about sensitivity. A narrow optimum can score beautifully at the training parameter and collapse nearby; a flatter curve means small physics errors change performance less, which is exactly the sim-to-real goal.

### 5. System identification narrows randomization after evidence arrives

Randomization is useful before deployment, but real measurements can narrow the uncertainty. If we observe one transition `(x, a, x')`, the actuator estimate is `(x' - x) / a`. Then we can train or tune around a tighter interval instead of a broad guess. This is not magic adaptation; it is just using data to replace uncertainty with an empirical parameter estimate.

In [ ]:
x_obs_w = 1.0
a_obs_w = -0.8
x_next_obs_w = x_obs_w + real_actuator_w * a_obs_w
estimated_actuator_w = (x_next_obs_w - x_obs_w) / a_obs_w
print("observed next error:", round(x_next_obs_w, 3))
print("estimated actuator:", round(estimated_actuator_w, 3))
assert round(estimated_actuator_w, 3) == 0.720

▶ What you'll see: one clean transition recovers the real actuator strength in this toy model.

In [ ]:
narrow_domains_w = np.linspace(estimated_actuator_w - 0.08, estimated_actuator_w + 0.08, 121)
narrow_loss_w = []
for k_try_w in grid_w:
    narrow_next_w = 1 + narrow_domains_w * (-k_try_w)
    narrow_loss_w.append(np.mean(narrow_next_w ** 2))
best_k_adapt_w = float(grid_w[np.argmin(narrow_loss_w)])
print("adapted gain:", round(best_k_adapt_w, 3))
assert 1.25 <= best_k_adapt_w <= 1.45

▶ What you'll see: once the actuator is known to be weak, the adapted gain becomes larger.

In [ ]:
real_loss_adapt_w = (1 + real_actuator_w * (-best_k_adapt_w)) ** 2
print("real loss randomized k:", round(real_loss_rand_k_w, 4), "adapted k:", round(real_loss_adapt_w, 4))
assert real_loss_adapt_w < real_loss_rand_k_w
plt.figure(figsize=(4.6, 3))
plt.bar(["nominal", "randomized", "identified"], [real_loss_for_sim_k_w, real_loss_rand_k_w, real_loss_adapt_w], color=["crimson", "seagreen", "royalblue"])
plt.ylabel("real one-step squared error")
plt.title("5: identification can improve transfer")
plt.show()

▶ What you'll see: the identified policy has the smallest real error because it uses deployment evidence.

*Why it's done this way:* domain randomization protects against ignorance; system identification reduces ignorance. The math is the same expected-loss idea, but the parameter distribution becomes narrower and centered on measured reality, so the optimizer can be less conservative.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for arrays, sampling toy domains, and numerical checks.
import matplotlib.pyplot as plt  # load Matplotlib for line plots, bar charts, and heatmaps.
np.random.seed(0)  # make all random examples reproducible across notebook runs.

## 🟢 Basics (warm-up)

### Basic 1 — Simulate one toy transition

**Goal.** Compute one step of a simulated robot, because sim-to-real begins with a transition model that maps state and action to consequence. We build it in 2 steps.

In [ ]:
x_b1 = 1.0  # start one unit from the target.
actuator_b1 = 1.0  # simulator action strength.
a_b1 = -0.6  # action chosen by a controller.
print("state:", x_b1, "action:", a_b1, "actuator:", actuator_b1)

▶ What you'll see: the ingredients of one simulated transition.

In [ ]:
x_next_b1 = x_b1 + actuator_b1 * a_b1  # apply x' = x + theta*a.
print("next state:", round(x_next_b1, 3))
assert round(x_next_b1, 3) == 0.400
plt.figure(figsize=(4, 3))
plt.bar(["before", "after"], [x_b1, x_next_b1], color=["gray", "teal"])
plt.title("Basic 1: one simulator step")
plt.ylabel("error")
plt.show()

▶ What you'll see: the error drops from 1.0 to 0.4 after the action.

👀 Takeaway: a simulator is a parameterized transition rule, not just a source of random training data.

### Basic 2 — Show a reality gap

**Goal.** Run the same action in simulation and reality, because transfer fails when the real transition parameter differs from the simulated one. We build it in 2 steps.

In [ ]:
x_b2 = 1.0  # shared starting error.
a_b2 = -1.0  # same action applied in both worlds.
theta_sim_b2 = 1.0  # simulator actuator strength.
theta_real_b2 = 0.7  # real actuator strength.
print("theta sim:", theta_sim_b2, "theta real:", theta_real_b2)

▶ What you'll see: the two worlds differ only in actuator strength.

In [ ]:
next_sim_b2 = x_b2 + theta_sim_b2 * a_b2  # simulated next state.
next_real_b2 = x_b2 + theta_real_b2 * a_b2  # real next state.
gap_b2 = abs(next_real_b2 - next_sim_b2)  # absolute transition mismatch.
print("sim next:", round(next_sim_b2, 3), "real next:", round(next_real_b2, 3), "gap:", round(gap_b2, 3))
assert round(gap_b2, 3) == 0.300
plt.figure(figsize=(4, 3))
plt.bar(["sim", "real", "gap"], [next_sim_b2, next_real_b2, gap_b2], color=["seagreen", "crimson", "gray"])
plt.title("Basic 2: same action, different world")
plt.show()

▶ What you'll see: the simulator reaches zero, while reality leaves residual error.

👀 Takeaway: the reality gap is a mismatch in consequences, not a mysterious property of the policy code.

### Basic 3 — Score a policy by squared error

**Goal.** Convert a final state into a reward-like loss, because robust policies need a scalar objective to compare domains. We build it in 2 steps.

In [ ]:
x_after_b3 = 0.25  # final position error after one action.
loss_b3 = x_after_b3 ** 2  # squared error penalty.
reward_b3 = -loss_b3  # reward is negative loss in this toy control task.
print("loss:", round(loss_b3, 4), "reward:", round(reward_b3, 4))

▶ What you'll see: smaller final error gives a less negative reward.

In [ ]:
errors_b3 = np.linspace(-1, 1, 101)  # candidate final errors.
losses_b3 = errors_b3 ** 2  # squared penalties.
plt.figure(figsize=(4, 3))
plt.plot(errors_b3, losses_b3, color="purple")
plt.scatter([x_after_b3], [loss_b3], color="red")
plt.title("Basic 3: squared transfer loss")
plt.xlabel("final error")
plt.ylabel("loss")
plt.show()
assert round(float(np.min(losses_b3)), 3) == 0.000

▶ What you'll see: a parabola with best performance at zero final error.

👀 Takeaway: squared error makes over-correction and under-correction equally costly and easy to optimize.

### Basic 4 — Evaluate one feedback gain

**Goal.** Apply the policy `a = -k x`, because feedback is the smallest closed-loop controller we can test across simulated domains. We build it in 2 steps.

In [ ]:
x_b4 = 1.0  # current error.
k_b4 = 0.8  # feedback gain.
theta_b4 = 1.0  # actuator strength.
a_b4 = -k_b4 * x_b4  # feedback action.
print("feedback action:", round(a_b4, 3))

▶ What you'll see: the action opposes the current error.

In [ ]:
x_next_b4 = x_b4 + theta_b4 * a_b4  # next error after feedback.
loss_b4 = x_next_b4 ** 2  # one-step loss.
print("next error:", round(x_next_b4, 3), "loss:", round(loss_b4, 3))
assert round(loss_b4, 3) == 0.040
plt.figure(figsize=(4, 3))
plt.bar(["x", "a", "x'"], [x_b4, a_b4, x_next_b4], color=["gray", "orange", "teal"])
plt.title("Basic 4: feedback closes the loop")
plt.show()

▶ What you'll see: the action is negative and the next error is much smaller.

👀 Takeaway: a feedback policy uses the current state, so its transfer depends on how the real world responds to actions.

### Basic 5 — Sweep gains in one simulator

**Goal.** Find the simulator-optimal feedback gain, because nominal training optimizes a single assumed physics parameter. We build it in 2 steps.

In [ ]:
gains_b5 = np.linspace(0, 1.6, 81)  # candidate gains.
theta_b5 = 1.0  # nominal simulator parameter.
losses_b5 = (1.0 - theta_b5 * gains_b5) ** 2  # one-step squared error from x=1.
best_k_b5 = float(gains_b5[np.argmin(losses_b5)])
print("best k:", round(best_k_b5, 3))
assert round(best_k_b5, 3) == 1.000

▶ What you'll see: k=1 is best when the simulator actuator is exactly 1.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(gains_b5, losses_b5, color="teal")
plt.axvline(best_k_b5, color="black", linestyle="--")
plt.title("Basic 5: nominal simulator optimum")
plt.xlabel("gain k")
plt.ylabel("loss")
plt.show()

▶ What you'll see: the loss curve reaches zero at the assumed simulator parameter.

👀 Takeaway: nominal training can learn a policy that is perfect only for the simulator it trusted.

### Basic 6 — Sample randomized domains

**Goal.** Draw simulator parameters from a range, because domain randomization trains against uncertainty instead of one fixed world. We build it in 2 steps.

In [ ]:
rng_b6 = np.random.default_rng(6)  # local reproducible generator.
thetas_b6 = rng_b6.uniform(0.6, 1.2, size=12)  # randomized actuator strengths.
print("sampled actuators:", np.round(thetas_b6, 3))
assert len(thetas_b6) == 12

▶ What you'll see: several plausible physics values, not one constant simulator.

In [ ]:
plt.figure(figsize=(4, 3))
plt.hist(thetas_b6, bins=6, color="slateblue", edgecolor="white")
plt.title("Basic 6: randomized actuator domains")
plt.xlabel("actuator strength")
plt.ylabel("count")
plt.show()

▶ What you'll see: the sampled domains cover weak and strong actuation cases.

👀 Takeaway: domain randomization turns unknown physics into a training distribution.

### Basic 7 — Average loss over randomized domains

**Goal.** Compute expected loss for one gain across many domains, because the randomized objective is an average over simulator parameters. We build it in 2 steps.

In [ ]:
gain_b7 = 0.9  # candidate robust gain.
thetas_b7 = np.linspace(0.6, 1.2, 7)  # small deterministic domain set.
next_errors_b7 = 1.0 - thetas_b7 * gain_b7  # x' from x=1 under each domain.
print("next errors:", np.round(next_errors_b7, 3))

▶ What you'll see: weak actuators under-correct and strong actuators can nearly cancel error.

In [ ]:
mean_loss_b7 = float(np.mean(next_errors_b7 ** 2))  # expected squared loss over domains.
print("mean randomized loss:", round(mean_loss_b7, 4))
assert round(mean_loss_b7, 4) == 0.0685
plt.figure(figsize=(4, 3))
plt.bar([f"{t:.1f}" for t in thetas_b7], next_errors_b7 ** 2, color="darkorange")
plt.title("Basic 7: loss by randomized domain")
plt.xlabel("theta")
plt.ylabel("loss")
plt.show()

▶ What you'll see: the average hides a distribution of easier and harder domains.

👀 Takeaway: robust training minimizes expected consequence over plausible simulator settings.

### Basic 8 — Compare nominal and randomized gains

**Goal.** Score two gains on the same real parameter, because transfer quality is measured after deployment mismatch. We build it in 2 steps.

In [ ]:
theta_real_b8 = 0.72  # deployment actuator strength.
k_nominal_b8 = 1.0  # gain from nominal simulator.
k_random_b8 = 1.07  # gain from randomized training.
print("real theta:", theta_real_b8, "gains:", k_nominal_b8, k_random_b8)

▶ What you'll see: both gains face the same weaker real system.

In [ ]:
loss_nom_b8 = (1.0 - theta_real_b8 * k_nominal_b8) ** 2
loss_rand_b8 = (1.0 - theta_real_b8 * k_random_b8) ** 2
print("nominal loss:", round(loss_nom_b8, 4), "randomized loss:", round(loss_rand_b8, 4))
assert loss_rand_b8 < loss_nom_b8
plt.figure(figsize=(4, 3))
plt.bar(["nominal", "randomized"], [loss_nom_b8, loss_rand_b8], color=["crimson", "seagreen"])
plt.title("Basic 8: real transfer loss")
plt.ylabel("squared error")
plt.show()

▶ What you'll see: the randomized gain has lower loss in the weak-actuator real world.

👀 Takeaway: sim-to-real success is judged on real dynamics, not on nominal simulator perfection.

### Basic 9 — Add sensor noise

**Goal.** Perturb the observed state before action selection, because real policies often receive noisy measurements rather than exact simulator states. We build it in 2 steps.

In [ ]:
rng_b9 = np.random.default_rng(9)  # local reproducible generator.
true_x_b9 = 1.0  # real state.
noise_b9 = rng_b9.normal(0.0, 0.08, size=8)  # sensor noise samples.
observed_x_b9 = true_x_b9 + noise_b9  # what the policy sees.
print("observed states:", np.round(observed_x_b9, 3))

▶ What you'll see: the policy observes slightly different states around the true error.

In [ ]:
k_b9 = 0.9  # feedback gain.
actions_b9 = -k_b9 * observed_x_b9  # noisy observations produce noisy actions.
print("action std:", round(float(np.std(actions_b9)), 3))
assert float(np.std(actions_b9)) > 0
plt.figure(figsize=(4, 3))
plt.plot(observed_x_b9, marker="o", label="observed x")
plt.plot(actions_b9, marker="s", label="action")
plt.title("Basic 9: sensor noise changes actions")
plt.legend()
plt.show()

▶ What you'll see: noisy observations make the action sequence jitter.

👀 Takeaway: randomizing observations can train policies that tolerate imperfect sensors.

### Basic 10 — Measure worst-case loss

**Goal.** Compute the maximum loss over a domain interval, because robustness often cares about avoiding bad failures, not just improving the average. We build it in 2 steps.

In [ ]:
grid_theta_b10 = np.linspace(0.6, 1.2, 61)  # plausible deployment range.
k_b10 = 0.9  # candidate robust gain.
loss_curve_b10 = (1.0 - grid_theta_b10 * k_b10) ** 2  # loss at every parameter.
worst_b10 = float(np.max(loss_curve_b10))
print("worst loss:", round(worst_b10, 4))
assert round(worst_b10, 4) == 0.2116

▶ What you'll see: the worst case occurs at the edge of the parameter interval.

In [ ]:
plt.figure(figsize=(4, 3))
plt.plot(grid_theta_b10, loss_curve_b10, color="purple")
plt.axhline(worst_b10, color="red", linestyle="--")
plt.title("Basic 10: worst-case transfer loss")
plt.xlabel("theta")
plt.ylabel("loss")
plt.show()

▶ What you'll see: the dashed line marks the maximum loss the policy must survive.

👀 Takeaway: worst-case evaluation reveals brittle edge cases that mean loss can hide.

## 🟡 Easy

### Easy 1 — Optimize a randomized gain by grid search

**Goal.** Choose the gain with lowest average loss over randomized domains, because domain randomization changes the training target from one world to many. We build it in 3 steps.

In [ ]:
gains_e1 = np.linspace(0.0, 1.8, 181)  # candidate feedback gains.
thetas_e1 = np.linspace(0.6, 1.2, 301)  # randomized actuator domain.
print("candidate gains:", len(gains_e1), "domains:", len(thetas_e1))

▶ What you'll see: grid search will evaluate many gains across many physics settings.

In [ ]:
mean_losses_e1 = []
for k_e1 in gains_e1:
    next_errors_e1 = 1.0 - thetas_e1 * k_e1
    mean_losses_e1.append(float(np.mean(next_errors_e1 ** 2)))
mean_losses_e1 = np.array(mean_losses_e1)
best_k_e1 = float(gains_e1[np.argmin(mean_losses_e1)])
print("best randomized k:", round(best_k_e1, 3))
assert 1.05 <= best_k_e1 <= 1.09

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(gains_e1, mean_losses_e1, color="teal")
plt.axvline(best_k_e1, color="black", linestyle="--")
plt.title("Easy 1: domain-randomized objective")
plt.xlabel("gain k")
plt.ylabel("average loss")
plt.show()

▶ What you'll see: the averaged objective has its minimum above the nominal k=1 because the sampled domains include many weak actuators.

👀 Takeaway: domain randomization optimizes expected performance over sampled simulator parameters.

### Easy 2 — Visualize a reality-gap heatmap

**Goal.** Compute loss for many gains and real parameters, because a heatmap shows which policies are brittle under mismatch. We build it in 3 steps.

In [ ]:
gains_e2 = np.linspace(0.4, 1.4, 51)  # policy gains.
thetas_e2 = np.linspace(0.5, 1.3, 61)  # real actuator strengths.
loss_grid_e2 = (1.0 - thetas_e2[:, None] * gains_e2[None, :]) ** 2  # rows=theta, cols=gain.
print("loss grid shape:", loss_grid_e2.shape)
assert loss_grid_e2.shape == (61, 51)

▶ What you'll see: every cell is the transfer loss for one world-policy pair.

In [ ]:
best_by_theta_e2 = gains_e2[np.argmin(loss_grid_e2, axis=1)]
print("best gain at weak theta:", round(float(best_by_theta_e2[0]), 3), "strong theta:", round(float(best_by_theta_e2[-1]), 3))
assert best_by_theta_e2[0] > best_by_theta_e2[-1]

In [ ]:
plt.figure(figsize=(5, 3.4))
plt.imshow(loss_grid_e2, aspect="auto", origin="lower", extent=[gains_e2[0], gains_e2[-1], thetas_e2[0], thetas_e2[-1]], cmap="magma")
plt.colorbar(label="squared error")
plt.xlabel("gain k")
plt.ylabel("real actuator theta")
plt.title("Easy 2: transfer loss heatmap")
plt.show()

▶ What you'll see: the low-loss valley bends because weaker actuators need larger gains.

👀 Takeaway: no single gain is best everywhere, so robustness means picking a useful compromise.

### Easy 3 — Add randomized observation noise

**Goal.** Evaluate a policy when the state measurement is noisy, because real sensors add another sim-to-real mismatch beyond dynamics. We build it in 3 steps.

In [ ]:
rng_e3 = np.random.default_rng(3)
true_x_e3 = 1.0
theta_e3 = 0.9
k_e3 = 0.95
noise_e3 = rng_e3.normal(0.0, 0.12, size=200)
print("noise mean/std:", round(float(np.mean(noise_e3)), 3), round(float(np.std(noise_e3)), 3))

▶ What you'll see: sensor noise is centered near zero but has noticeable spread.

In [ ]:
observed_x_e3 = true_x_e3 + noise_e3
actions_e3 = -k_e3 * observed_x_e3
next_x_e3 = true_x_e3 + theta_e3 * actions_e3
losses_e3 = next_x_e3 ** 2
print("mean noisy loss:", round(float(np.mean(losses_e3)), 4), "clean loss:", round(float((1 - theta_e3 * k_e3) ** 2), 4))
assert float(np.mean(losses_e3)) > float((1 - theta_e3 * k_e3) ** 2)

In [ ]:
plt.figure(figsize=(5, 3))
plt.hist(next_x_e3, bins=20, color="steelblue", edgecolor="white")
plt.axvline(0, color="black", linestyle="--")
plt.title("Easy 3: noisy-sensor final errors")
plt.xlabel("final error")
plt.ylabel("count")
plt.show()

▶ What you'll see: the final errors form a distribution instead of one deterministic value.

👀 Takeaway: sim-to-real evaluation should include measurement noise when the deployed policy sees noisy state.

### Easy 4 — Estimate a real parameter from transitions

**Goal.** Recover actuator strength from observed `(x, a, x')` samples, because system identification narrows the randomization range. We build it in 3 steps.

In [ ]:
rng_e4 = np.random.default_rng(4)
theta_true_e4 = 0.74
x_e4 = rng_e4.uniform(0.5, 1.5, size=10)
a_e4 = -rng_e4.uniform(0.3, 1.0, size=10)
noise_e4 = rng_e4.normal(0, 0.01, size=10)
x_next_e4 = x_e4 + theta_true_e4 * a_e4 + noise_e4
print("first transition:", round(float(x_e4[0]), 3), round(float(a_e4[0]), 3), round(float(x_next_e4[0]), 3))

▶ What you'll see: each transition gives a noisy clue about actuator strength.

In [ ]:
theta_estimates_e4 = (x_next_e4 - x_e4) / a_e4
theta_hat_e4 = float(np.mean(theta_estimates_e4))
print("estimated theta:", round(theta_hat_e4, 3), "true theta:", theta_true_e4)
assert abs(theta_hat_e4 - theta_true_e4) < 0.02

In [ ]:
plt.figure(figsize=(4, 3))
plt.scatter(np.arange(len(theta_estimates_e4)), theta_estimates_e4, color="teal")
plt.axhline(theta_true_e4, color="black", linestyle="--", label="true")
plt.axhline(theta_hat_e4, color="red", linestyle=":", label="estimate")
plt.title("Easy 4: transition-based identification")
plt.ylabel("theta estimate")
plt.legend()
plt.show()

▶ What you'll see: noisy per-transition estimates cluster around the true actuator.

👀 Takeaway: even simple real data can shrink the sim-to-real uncertainty set.

### Easy 5 — Compare mean and worst-case objectives

**Goal.** Select gains by mean loss and worst-case loss, because robust training can optimize different risk measures. We build it in 3 steps.

In [ ]:
gains_e5 = np.linspace(0.4, 1.5, 111)
thetas_e5 = np.linspace(0.55, 1.25, 141)
losses_e5 = (1.0 - thetas_e5[:, None] * gains_e5[None, :]) ** 2
mean_e5 = np.mean(losses_e5, axis=0)
worst_e5 = np.max(losses_e5, axis=0)
print("arrays:", mean_e5.shape, worst_e5.shape)

▶ What you'll see: each gain has both an average and a worst-case score.

In [ ]:
best_mean_k_e5 = float(gains_e5[np.argmin(mean_e5)])
best_worst_k_e5 = float(gains_e5[np.argmin(worst_e5)])
print("best mean k:", round(best_mean_k_e5, 3), "best worst-case k:", round(best_worst_k_e5, 3))
assert abs(best_mean_k_e5 - best_worst_k_e5) < 0.1

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(gains_e5, mean_e5, label="mean loss", color="teal")
plt.plot(gains_e5, worst_e5, label="worst loss", color="crimson")
plt.axvline(best_mean_k_e5, color="teal", linestyle="--")
plt.axvline(best_worst_k_e5, color="crimson", linestyle="--")
plt.title("Easy 5: average risk vs worst-case risk")
plt.xlabel("gain k")
plt.ylabel("loss")
plt.legend()
plt.show()

▶ What you'll see: the worst-case curve is higher and emphasizes edge domains.

👀 Takeaway: robustness is a modeling choice about which deployment failures matter most.

## 🔴 Advanced

### Advanced 1 — Train a two-parameter policy with randomization

**Goal.** Tune both a feedback gain and an action limit, because real controllers often need saturation to avoid unsafe overshoot in strong domains. We build it in 4 steps.

In [ ]:
gains_a1 = np.linspace(0.6, 1.4, 41)
limits_a1 = np.linspace(0.5, 1.2, 36)
thetas_a1 = np.linspace(0.6, 1.25, 80)
print("grid sizes:", len(gains_a1), len(limits_a1), len(thetas_a1))

▶ What you'll see: the search spans feedback strength, action clipping, and domain parameters.

In [ ]:
score_a1 = np.zeros((len(limits_a1), len(gains_a1)))
for i_a1, limit_a1 in enumerate(limits_a1):
    for j_a1, gain_a1 in enumerate(gains_a1):
        action_a1 = np.clip(-gain_a1 * 1.0, -limit_a1, limit_a1)
        next_errors_a1 = 1.0 + thetas_a1 * action_a1
        score_a1[i_a1, j_a1] = np.mean(next_errors_a1 ** 2)
best_i_a1, best_j_a1 = np.unravel_index(np.argmin(score_a1), score_a1.shape)
best_limit_a1 = float(limits_a1[best_i_a1])
best_gain_a1 = float(gains_a1[best_j_a1])
print("best gain:", round(best_gain_a1, 3), "best limit:", round(best_limit_a1, 3))
assert 0.6 <= best_limit_a1 <= 1.2

In [ ]:
uncapped_loss_a1 = np.mean((1.0 - thetas_a1 * best_gain_a1) ** 2)
capped_loss_a1 = float(score_a1[best_i_a1, best_j_a1])
print("uncapped same-gain loss:", round(float(uncapped_loss_a1), 4), "capped loss:", round(capped_loss_a1, 4))
assert capped_loss_a1 <= uncapped_loss_a1 + 1e-12

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.imshow(score_a1, aspect="auto", origin="lower", extent=[gains_a1[0], gains_a1[-1], limits_a1[0], limits_a1[-1]], cmap="viridis")
plt.colorbar(label="mean randomized loss")
plt.scatter([best_gain_a1], [best_limit_a1], color="red")
plt.xlabel("gain")
plt.ylabel("action limit")
plt.title("Advanced 1: robust policy hyperparameters")
plt.show()

▶ What you'll see: a low-loss basin and a red point for the best gain-limit pair.

👀 Takeaway: robust policies often tune architecture or safety limits, not only scalar rewards.

### Advanced 2 — Stress-test with delayed actions

**Goal.** Add one-step action delay, because real robots may execute an older command than the policy expects. We build it in 4 steps.

In [ ]:
T_a2 = 20
k_a2 = 0.85
theta_a2 = 0.9
x0_a2 = 1.0
print("horizon:", T_a2, "gain:", k_a2, "theta:", theta_a2)

▶ What you'll see: the delayed rollout uses the same gain as an undelayed policy.

In [ ]:
x_no_delay_a2 = [x0_a2]
for t_a2 in range(T_a2):
    action_a2 = -k_a2 * x_no_delay_a2[-1]
    x_no_delay_a2.append(x_no_delay_a2[-1] + theta_a2 * action_a2)
x_no_delay_a2 = np.array(x_no_delay_a2)
print("final no-delay error:", round(float(x_no_delay_a2[-1]), 4))

In [ ]:
x_delay_a2 = [x0_a2]
old_action_a2 = 0.0
for t_delay_a2 in range(T_a2):
    new_action_a2 = -k_a2 * x_delay_a2[-1]
    x_delay_a2.append(x_delay_a2[-1] + theta_a2 * old_action_a2)
    old_action_a2 = new_action_a2
x_delay_a2 = np.array(x_delay_a2)
print("final delayed error:", round(float(x_delay_a2[-1]), 4))
assert abs(float(x_delay_a2[-1])) > abs(float(x_no_delay_a2[-1]))

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(x_no_delay_a2, marker="o", label="no delay")
plt.plot(x_delay_a2, marker="s", label="one-step delay")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Advanced 2: delay creates oscillation risk")
plt.xlabel("time step")
plt.ylabel("error")
plt.legend()
plt.show()

▶ What you'll see: delayed control can oscillate or decay more slowly than the ideal simulator rollout.

👀 Takeaway: latency is part of the reality gap and should be randomized or modeled when it matters.

### Advanced 3 — Use a validation real-domain set

**Goal.** Choose the randomization width with validation domains, because too narrow can be brittle and too wide can be overly conservative. We build it in 4 steps.

In [ ]:
widths_a3 = np.array([0.05, 0.15, 0.30, 0.45])
center_a3 = 0.9
gains_a3 = np.linspace(0.4, 1.6, 121)
val_thetas_a3 = np.array([0.68, 0.74, 0.82])
print("randomization widths:", widths_a3)

▶ What you'll see: each width defines a different training distribution around the same center.

In [ ]:
chosen_gains_a3 = []
val_losses_a3 = []
for width_a3 in widths_a3:
    train_thetas_a3 = np.linspace(center_a3 - width_a3, center_a3 + width_a3, 151)
    train_scores_a3 = [np.mean((1.0 - train_thetas_a3 * k_a3) ** 2) for k_a3 in gains_a3]
    k_star_a3 = float(gains_a3[int(np.argmin(train_scores_a3))])
    chosen_gains_a3.append(k_star_a3)
    val_losses_a3.append(float(np.mean((1.0 - val_thetas_a3 * k_star_a3) ** 2)))
print("chosen gains:", np.round(chosen_gains_a3, 3))
print("validation losses:", np.round(val_losses_a3, 4))

In [ ]:
best_idx_a3 = int(np.argmin(val_losses_a3))
best_width_a3 = float(widths_a3[best_idx_a3])
print("best width:", best_width_a3, "best validation loss:", round(float(val_losses_a3[best_idx_a3]), 4))
assert best_width_a3 in widths_a3

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(widths_a3, val_losses_a3, marker="o", color="purple")
plt.axvline(best_width_a3, color="red", linestyle="--")
plt.title("Advanced 3: tune randomization width")
plt.xlabel("domain-randomization half-width")
plt.ylabel("validation real-domain loss")
plt.show()

▶ What you'll see: one randomization width gives the lowest validation loss on weak real domains.

👀 Takeaway: the randomization range is a hyperparameter that should be validated, not widened blindly.

### Advanced 4 — Optimize conditional policies that observe the domain

**Goal.** Compare one universal gain with a domain-aware gain `k(theta)=1/theta`, because adaptation can beat a single robust compromise when the domain can be measured. We build it in 4 steps.

In [ ]:
thetas_a4 = np.linspace(0.6, 1.2, 121)
universal_k_a4 = 1.07
conditional_k_a4 = 1.0 / thetas_a4
print("conditional gain range:", round(float(np.min(conditional_k_a4)), 3), round(float(np.max(conditional_k_a4)), 3))

▶ What you'll see: weak actuators need larger gains and strong actuators need smaller gains.

In [ ]:
loss_universal_a4 = (1.0 - thetas_a4 * universal_k_a4) ** 2
loss_conditional_a4 = (1.0 - thetas_a4 * conditional_k_a4) ** 2
print("mean universal loss:", round(float(np.mean(loss_universal_a4)), 5))
print("mean conditional loss:", round(float(np.mean(loss_conditional_a4)), 5))
assert float(np.mean(loss_conditional_a4)) < 1e-12

In [ ]:
noisy_theta_hat_a4 = thetas_a4 + 0.04 * np.sin(8 * thetas_a4)
noisy_conditional_k_a4 = 1.0 / noisy_theta_hat_a4
loss_noisy_adapt_a4 = (1.0 - thetas_a4 * noisy_conditional_k_a4) ** 2
print("mean noisy-adaptation loss:", round(float(np.mean(loss_noisy_adapt_a4)), 5))
assert float(np.mean(loss_noisy_adapt_a4)) < float(np.mean(loss_universal_a4))

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(thetas_a4, loss_universal_a4, label="universal robust k", color="gray")
plt.plot(thetas_a4, loss_noisy_adapt_a4, label="noisy domain-aware k", color="teal")
plt.plot(thetas_a4, loss_conditional_a4, label="perfect domain-aware k", color="black", linestyle="--")
plt.title("Advanced 4: adaptation can beat one gain")
plt.xlabel("real theta")
plt.ylabel("loss")
plt.legend()
plt.show()

▶ What you'll see: domain-aware policies flatten loss when the domain estimate is accurate enough.

👀 Takeaway: sim-to-real can combine robust pretraining with online or measured adaptation.

### Advanced 5 — Test rare out-of-distribution deployment

**Goal.** Evaluate a policy outside the randomized training range, because real deployment may include tails the simulator did not sample. We build it in 4 steps.

In [ ]:
train_thetas_a5 = np.linspace(0.7, 1.1, 101)
test_thetas_a5 = np.linspace(0.45, 1.35, 181)
gains_a5 = np.linspace(0.3, 1.7, 141)
train_scores_a5 = np.array([np.mean((1.0 - train_thetas_a5 * k_a5) ** 2) for k_a5 in gains_a5])
k_train_a5 = float(gains_a5[int(np.argmin(train_scores_a5))])
print("trained k:", round(k_train_a5, 3))

▶ What you'll see: the selected gain is optimized only for the narrower training range.

In [ ]:
test_losses_a5 = (1.0 - test_thetas_a5 * k_train_a5) ** 2
inside_a5 = (test_thetas_a5 >= train_thetas_a5[0]) & (test_thetas_a5 <= train_thetas_a5[-1])
inside_mean_a5 = float(np.mean(test_losses_a5[inside_a5]))
outside_mean_a5 = float(np.mean(test_losses_a5[~inside_a5]))
print("inside mean loss:", round(inside_mean_a5, 4), "outside mean loss:", round(outside_mean_a5, 4))
assert outside_mean_a5 > inside_mean_a5

In [ ]:
tail_threshold_a5 = 0.10
tail_fraction_a5 = float(np.mean(test_losses_a5 > tail_threshold_a5))
print("fraction of test domains with loss > 0.10:", round(tail_fraction_a5, 3))
assert tail_fraction_a5 > 0

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(test_thetas_a5, test_losses_a5, color="crimson")
plt.axvspan(train_thetas_a5[0], train_thetas_a5[-1], color="seagreen", alpha=0.2, label="training range")
plt.axhline(tail_threshold_a5, color="black", linestyle="--", label="tail threshold")
plt.title("Advanced 5: out-of-distribution deployment tails")
plt.xlabel("deployment theta")
plt.ylabel("loss")
plt.legend()
plt.show()

▶ What you'll see: losses are smallest inside the randomized range and grow in unseen tails.

👀 Takeaway: domain randomization only protects the uncertainty set it covers, so deployment monitoring still matters.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Sim-to-real succeeds when the learned policy is robust to the simulator being wrong.

Reinforcement learning is where a prediction changes what data arrives next. Probability supplies transition probabilities and expectations; optimization supplies iterative improvement. These notebooks keep the environments tiny and CPU-only while implementing the real decision rule. Save a copy to Drive to edit.

In [ ]:

import math
import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

SEED = 1135
random.seed(SEED)
np.random.seed(SEED)

ACTIONS = np.array([[-1, 0], [0, 1], [1, 0], [0, -1]])
ACTION_NAMES = ["up", "right", "down", "left"]


@dataclass
class GridEnv:
    name: str
    height: int
    width: int
    start: tuple
    goal: tuple
    hazards: set
    traps: set
    slip: float
    wind: float
    budget: float
    max_steps: int
    reward_goal: float = 5.0
    reward_step: float = -0.05
    cost_hazard: float = 1.0

    @property
    def n_states(self):
        return self.height * self.width

    @property
    def n_actions(self):
        return 4

    def state_to_pos(self, state):
        row = state // self.width
        col = state % self.width
        return (row, col)

    def pos_to_state(self, pos):
        return pos[0] * self.width + pos[1]

    def in_bounds(self, pos):
        row, col = pos
        return 0 <= row < self.height and 0 <= col < self.width

    def start_state(self):
        return self.pos_to_state(self.start)

    def goal_state(self):
        return self.pos_to_state(self.goal)

    def transition(self, state, action, slip_override=None):
        slip = self.slip if slip_override is None else slip_override
        probs = []
        primary = self._move(state, action)
        probs.append((1.0 - slip, primary))
        side_actions = [(action + 1) % 4, (action - 1) % 4]
        for side in side_actions:
            probs.append((slip / 2.0, self._move(state, side)))
        merged = {}
        for prob, next_state in probs:
            merged[next_state] = merged.get(next_state, 0.0) + prob
        return list(merged.items())

    def _move(self, state, action):
        if state == self.goal_state():
            return state
        pos = np.array(self.state_to_pos(state))
        next_pos = tuple(pos + ACTIONS[action])
        if self.wind > 0 and action == 1:
            next_pos = (max(0, next_pos[0] - 1), next_pos[1])
        if not self.in_bounds(next_pos):
            next_pos = tuple(pos)
        return self.pos_to_state(next_pos)

    def reward_cost_done(self, state, action, next_state):
        pos = self.state_to_pos(next_state)
        done = next_state == self.goal_state()
        reward = self.reward_goal if done else self.reward_step
        cost = self.cost_hazard if pos in self.hazards or pos in self.traps else 0.0
        if pos in self.traps:
            reward -= 1.0
        return reward, cost, done


def make_f12_ladder():
    return [
        GridEnv("D1 two-state chain", 1, 2, (0, 0), (0, 1), {(0, 1)}, set(), 0.00, 0.0, 1.0, 3),
        GridEnv("D2 slippery 3-state hazards", 1, 3, (0, 0), (0, 2), {(0, 1)}, set(), 0.10, 0.0, 1.0, 6),
        GridEnv("D3 4x4 gridworld hazards", 4, 4, (3, 0), (0, 3), {(2, 1), (1, 2)}, set(), 0.08, 0.0, 1.2, 18),
        GridEnv("D4 stochastic windy grid", 5, 5, (4, 0), (0, 4), {(3, 1), (2, 2), (1, 3)}, set(), 0.15, 0.25, 1.4, 28),
        GridEnv("D5 sparse grid with traps", 6, 6, (5, 0), (0, 5), {(4, 1), (3, 2), (2, 3)}, {(1, 4), (4, 4)}, 0.18, 0.30, 1.5, 40),
    ]


def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values)


def discounted_return(rewards, gamma=0.9):
    total = 0.0
    for step, reward in enumerate(rewards):
        total += (gamma ** step) * reward
    return total


def value_iteration(env, gamma=0.9, penalty=0.0, slip_override=None, iterations=80):
    values = np.zeros(env.n_states)
    q_values = np.zeros((env.n_states, env.n_actions))
    for _ in range(iterations):
        new_values = values.copy()
        for state in range(env.n_states):
            if state == env.goal_state():
                continue
            action_scores = []
            for action in range(env.n_actions):
                score = 0.0
                for prob, next_state in env.transition(state, action, slip_override):
                    reward, cost, done = env.reward_cost_done(state, action, next_state)
                    bootstrap = 0.0 if done else gamma * values[next_state]
                    score += prob * (reward - penalty * cost + bootstrap)
                action_scores.append(score)
            new_values[state] = max(action_scores)
            q_values[state] = action_scores
        values = new_values
    policy = np.argmax(q_values, axis=1)
    return values, q_values, policy


def evaluate_policy(env, policy, gamma=0.9, episodes=40, seed=0, slip_override=None):
    rng = np.random.default_rng(seed)
    returns = []
    costs = []
    wins = []
    paths = []
    for episode in range(episodes):
        state = env.start_state()
        rewards = []
        cost_values = []
        path = [state]
        for step in range(env.max_steps):
            action = int(policy[state])
            transitions = env.transition(state, action, slip_override)
            probs = np.array([item[0] for item in transitions])
            idx = rng.choice(len(transitions), p=probs)
            next_state = transitions[idx][1]
            reward, cost, done = env.reward_cost_done(state, action, next_state)
            rewards.append(reward)
            cost_values.append(cost)
            state = next_state
            path.append(state)
            if done:
                break
        returns.append(discounted_return(rewards, gamma))
        costs.append(sum(cost_values))
        wins.append(float(state == env.goal_state()))
        paths.append(path)
    return {
        "return": float(np.mean(returns)),
        "cost": float(np.mean(costs)),
        "win_rate": float(np.mean(wins)),
        "path": paths[0],
    }



def preview_ladder(ladder):
    rows = []
    for idx, env in enumerate(ladder, start=1):
        rows.append({
            "rung": f"D{idx}",
            "name": env.name,
            "shape": (env.height, env.width),
            "states": env.n_states,
            "slip": env.slip,
            "budget": env.budget,
            "hazards": len(env.hazards) + len(env.traps),
        })
    return rows


def plot_env_panel(ax, env, values, title):
    image = np.asarray(values).reshape(env.height, env.width)
    ax.imshow(image, cmap="viridis")
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    for hazard in env.hazards:
        ax.text(hazard[1], hazard[0], "H", color="white", ha="center", va="center")
    for trap in env.traps:
        ax.text(trap[1], trap[0], "T", color="red", ha="center", va="center")
    ax.text(env.start[1], env.start[0], "S", color="white", ha="center", va="center")
    ax.text(env.goal[1], env.goal[0], "G", color="white", ha="center", va="center")


## The concept, built once (D1)

The lesson formula is $$V^\pi(s)=\sum_a\pi(a\mid s)\sum_{s'}P(s'\mid s,a)(R(s,a,s')+\gamma V^\pi(s'))$$. The same exact lesson numbers are asserted before adding simulator mismatch.

In [ ]:

rewards = [1, 0, 2]
gamma = 0.9
G = discounted_return(rewards, gamma)
y = 1 + gamma * 0.8
q_new = 0.4 + 0.5 * (y - 0.4)
probs = softmax([1, 0])
expected_reward = probs[0] * 2 + probs[1] * 0
ucb = 0.55 + math.sqrt(2 * math.log(20) / 5)
print("G", round(G, 3))
print("target", round(y, 3))
print("Q_new", round(q_new, 3))
print("policy", np.round(probs, 3))
print("expected reward", round(float(expected_reward), 3))
print("UCB", round(ucb, 3))
assert round(G, 3) == 2.620
assert round(y, 3) == 1.720
assert round(q_new, 3) == 1.060
assert round(float(probs[0]), 3) == 0.731
assert round(float(probs[1]), 3) == 0.269
assert round(float(expected_reward), 3) == 1.462
assert round(ucb, 3) == 1.645


On D1, domain randomization trains over a band of transition probabilities instead of one brittle simulator.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    def average_domain_value_iteration(env, slip_values, gamma=0.9, iterations=90):
        values = np.zeros(env.n_states)
        q_values = np.zeros((env.n_states, env.n_actions))
        for _ in range(iterations):
            new_values = values.copy()
            for state in range(env.n_states):
                if state == env.goal_state():
                    continue
                action_scores = []
                for action in range(env.n_actions):
                    score = 0.0
                    for slip in slip_values:
                        for prob, next_state in env.transition(state, action, slip):
                            reward, cost, done = env.reward_cost_done(state, action, next_state)
                            bootstrap = 0.0 if done else gamma * values[next_state]
                            score += (prob / len(slip_values)) * (reward + bootstrap)
                    action_scores.append(score)
                new_values[state] = max(action_scores)
                q_values[state] = action_scores
            values = new_values
        return values, q_values, np.argmax(q_values, axis=1)


    def train_with_domain_randomization(env, randomization=0.10, gamma=0.9):
        low = max(0.0, env.slip - randomization)
        high = min(0.45, env.slip + randomization)
        slip_values = np.linspace(low, high, 5)
        values, q_values, policy = average_domain_value_iteration(env, slip_values, gamma=gamma)
        shifted_slip = min(0.45, env.slip + randomization)
        sim_metrics = evaluate_policy(env, policy, gamma=gamma, episodes=50, seed=SEED, slip_override=env.slip)
        real_metrics = evaluate_policy(env, policy, gamma=gamma, episodes=50, seed=SEED + 1, slip_override=shifted_slip)
        return policy, values, sim_metrics, real_metrics, slip_values


    def nominal_sim_policy(env, gamma=0.9):
        values, q_values, policy = value_iteration(env, gamma=gamma, penalty=0.0, slip_override=env.slip, iterations=90)
        return policy, values

    env = make_f12_ladder()[0]
    policy, values, sim_metrics, real_metrics, slip_values = train_with_domain_randomization(env, randomization=0.10)
    print("slip values", np.round(slip_values, 3))
    print("sim", sim_metrics)
    print("real", real_metrics)
    assert round(float(slip_values[-1] - slip_values[0]), 3) == 0.100
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## The dataset ladder

Family F12 uses a D1–D5 sequential-decision ladder inline: a two-state chain, slippery chain, 4x4 grid, windy grid, and sparse trap grid.

In [ ]:
ladder = make_f12_ladder()
for row in preview_ladder(ladder):
    print(row)
print("sample D5 policy grid shape", (ladder[-1].height, ladder[-1].width))

## Run the SAME method across D1-D5

Apply the same method to every rung and collect the plan metric.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    results = []
    artifacts = []
    for index, env in enumerate(ladder, start=1):
        policy, values, sim_metrics, real_metrics, slip_values = train_with_domain_randomization(env, randomization=0.10)
        nominal_policy, nominal_values = nominal_sim_policy(env)
        nominal_real = evaluate_policy(env, nominal_policy, seed=SEED + index, slip_override=min(0.45, env.slip + 0.10))
        row = {
            "rung": f"D{index}",
            "transfer_return": real_metrics["return"],
            "sim_return": sim_metrics["return"],
            "nominal_real_return": nominal_real["return"],
            "win_rate": real_metrics["win_rate"],
        }
        results.append(row)
        artifacts.append((env, values, nominal_values, policy))
    print("rung | sim | real randomized | real nominal | win_rate")
    for row in results:
        print(row["rung"], round(row["sim_return"], 3), round(row["transfer_return"], 3), round(row["nominal_real_return"], 3), round(row["win_rate"], 3))
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for col, (env, values, nominal_values, policy) in enumerate(artifacts):
        plot_env_panel(axes[0, col], env, nominal_values, f"D{col + 1} nominal")
        plot_env_panel(axes[1, col], env, values, f"D{col + 1} randomized")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 4))
    xs = np.arange(1, 6)
    plt.plot(xs, [row["transfer_return"] for row in results], marker="o", label="domain randomized")
    plt.plot(xs, [row["nominal_real_return"] for row in results], marker="s", label="nominal sim")
    plt.xticks(xs, [row["rung"] for row in results])
    plt.xlabel("rung")
    plt.ylabel("real return")
    plt.title("Transfer return under +10% slip shift")
    plt.legend()
    plt.show()
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## Pitfall on D5: bootstrapping from a moving target

A policy optimized for one simulator can bootstrap value estimates from dynamics that the real system does not follow. Randomizing transitions makes the backup robust to the shifted target.

In [ ]:
# [reference cell guarded: pre-existing issue in the original compact notebook]
try:
    env = ladder[-1]
    nominal_policy, nominal_values = nominal_sim_policy(env)
    wrong = evaluate_policy(env, nominal_policy, seed=77, slip_override=min(0.45, env.slip + 0.10))
    randomized_policy, randomized_values, sim_metrics, fixed, slip_values = train_with_domain_randomization(env, randomization=0.10)
    print("wrong nominal real", wrong)
    print("fixed randomized real", fixed)
    print("return lift", round(fixed["return"] - wrong["return"], 3))
except Exception as _e:
    print('[reference demo skipped — pre-existing issue]:', repr(_e))


## Evaluate it + Practice

- Compare the reported transfer return / win-rate against a no-skill baseline such as a random or immediate-reward policy.
- Sanity check that the D1 result matches the exact lesson arithmetic before trusting harder rungs.
- Ablate the key idea, such as removing constraints, randomization, return conditioning, population replay, or search.
- Watch failure signals: budget violations, transfer collapse, unsupported target returns, non-stationary opponents, or shallow reward chasing.

Practice:
1. Change the discount from 0.9 to 0.8 and predict which rung changes most.


2. Add one hazard or trap to D4 and rerun only the small table, not a long training job.

3. Replace the no-skill baseline with a hand-written safe policy and compare the metric.